In [ ]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax
import jax.numpy as jnp
import optax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_fly import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom_rs as bom
from models import hgm_rs as hgm


jax.config.update("jax_enable_x64", True)

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)

In [ ]:
default_hyperparameters = {
	'max_epochs': 3000,
	# 'max_epochs': 3000,
	'max_patience': 3000 / 4,
	'learning_rate': 1E-0,
	'dimensions': 2,
	'ccadiag_lambda': 0.6,
	'ccaunit_lambda': 1.0 - 0.6,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'vb_reg_lambda': 0.0,
	'wb_reg_lambda': 0.0,
	'wm_reg_lambda': 0.0,
}

# Functions

In [ ]:
class Lambdas(NamedTuple):
	loss_lambdas: tuple[float, float]
	reg_lambdas: tuple[float, float, float]
	cca_lambdas: tuple[float, float]

In [ ]:
def train_model(model, splits, N, B, S, hyperparams=None):
	#region hyperparamters ===============================================================
	seeds = jax.random.PRNGKey(0), jax.random.PRNGKey(1), jax.random.PRNGKey(2), jax.random.PRNGKey(3), jax.random.PRNGKey(4)

	global default_hyperparameters
	hyperparameters = default_hyperparameters.copy()
	if hyperparams is not None:
		for key in hyperparams:
			hyperparameters[key] = hyperparams[key]

	max_epochs = hyperparameters['max_epochs']
	max_patience = hyperparameters['max_patience']

	learning_rate = hyperparameters['learning_rate']

	n_dimensions = hyperparameters['dimensions']

	lambdas = Lambdas(
		loss_lambdas=(hyperparameters['loss_man_lambda'], hyperparameters['loss_res_lambda']),
		reg_lambdas=(hyperparameters['vb_reg_lambda'], hyperparameters['wb_reg_lambda'], hyperparameters['wm_reg_lambda']),
		cca_lambdas=(hyperparameters['ccadiag_lambda'], hyperparameters['ccaunit_lambda'])
	)
	#endregion ===========================================================================

	n_neurons = N.shape[1]
	n_states = S.shape[1]
	n_behaviors = B.shape[1]

	# training loop ==============================
	params_per_split = list()
	(losses_train_per_split,
	 man_losses_train_per_split,
	 res_losses_train_per_split,
	 mpe_losses_train_per_split) = list(), list(), list(), list()
	(losses_per_split,
	 man_losses_per_split,
	 res_losses_per_split,
	 mpe_losses_per_split) = list(), list(), list(), list()
	for s_id, (train, eval) in enumerate(splits):
		#region splitting and initializing (this also moves the data to the GPU) ==========================
		N_train, N_eval = jnp.array(N[train]), jnp.array(N[eval])
		S_train, S_eval = jnp.array(S[train]), jnp.array(S[eval])
		B_train, B_eval = jnp.array(B[train]), jnp.array(B[eval])

		params = model.init_params(N_train, B_train, S_train,
								   seeds,
								   (n_neurons, n_dimensions),
								   (n_states, n_dimensions),
								   (n_behaviors, n_neurons),
								   (n_behaviors, n_behaviors),
								   * ((n_dimensions, n_behaviors),) if model.get_type() else (),
								  )

		optimizer = optax.adam(learning_rate=optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=max_epochs, alpha=1E-4))
		# optimizer = optax.adam(learning_rate=learning_rate)
		optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
		opt_state = optimizer.init(params)
		#endregion ========================================================================================

		(losses_train,
		 man_losses_train,
		 res_losses_train,
		 mpe_losses_train) = [], [], [], []
		(losses,
		 man_losses,
		 res_losses,
		 mpe_losses) = [], [], [], []

		best_params = params
		(best_loss_train,
		 best_man_loss_train,
		 best_res_loss_train,
		 best_mpe_loss_train) = sys.float_info.max, sys.float_info.max, sys.float_info.max, sys.float_info.max
		(best_loss,
		 best_man_loss,
		 best_res_loss,
		 best_mpe_loss) = sys.float_info.max, sys.float_info.max, sys.float_info.max, sys.float_info.max

		# lr_step = 0
		patience = 0
		for epoch in range(max_epochs):

			(loss,
			 man_loss,
			 res_loss,
			 mpe_loss) = tuple(l.item() for l in model.loss_fn_flattened(params, N_eval, B_eval, S_eval, lambdas))

			(opt_state, params, loss_types_train) = model.update(optimizer.update, opt_state, params, N_train, B_train, S_train, lambdas)

			(loss_train,
			 man_loss_train,
			 res_loss_train,
			 mpe_loss_train) = tuple(l.item() for l in loss_types_train)

			#region early stopping ===========================================================================
			if epoch >= max_patience:  # give it an initial time
				if loss < best_loss:
					patience = 0

					best_params = params
					(best_loss_train,
					 best_man_loss_train,
					 best_res_loss_train,
					 best_mpe_loss_train) = loss_train, man_loss_train, res_loss_train, mpe_loss_train
					(best_loss,
					 best_man_loss,
					 best_res_loss,
					 best_mpe_loss) = loss, man_loss, res_loss, mpe_loss
				else:
					patience += 1
					if patience > max_patience:
						params = best_params

						losses_train = losses_train + [best_loss_train for _ in range(max_epochs - epoch)]
						man_losses_train = man_losses_train + [best_man_loss_train for _ in range(max_epochs - epoch)]
						res_losses_train = res_losses_train + [best_res_loss_train for _ in range(max_epochs - epoch)]
						mpe_losses_train = mpe_losses_train + [best_mpe_loss_train for _ in range(max_epochs - epoch)]

						losses = losses + [best_loss for _ in range(max_epochs - epoch)]
						man_losses = man_losses + [best_man_loss for _ in range(max_epochs - epoch)]
						res_losses = res_losses + [best_res_loss for _ in range(max_epochs - epoch)]
						mpe_losses = mpe_losses + [best_mpe_loss for _ in range(max_epochs - epoch)]

						print(f'Stopped after {epoch} epochs: ', end='')
						break

						# lr_step += 1
						# optimizer = optax.adam(learning_rate / (10 ** lr_step))
						# optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
						# patience = 0
						# print(f'\nReduced learning rate from {learning_rate / (10 ** (lr_step - 1))} to {learning_rate / (10 ** lr_step)} after epoch {epoch} in split {s_id}', end='')
			#endregion =======================================================================================

			losses_train.append(loss_train)
			man_losses_train.append(man_loss_train)
			res_losses_train.append(res_loss_train)
			mpe_losses_train.append(mpe_loss_train)

			losses.append(loss)
			man_losses.append(man_loss)
			res_losses.append(res_loss)
			mpe_losses.append(mpe_loss)

		params_per_split.append(params)

		losses_train_per_split.append(losses_train)
		man_losses_train_per_split.append(man_losses_train)
		res_losses_train_per_split.append(res_losses_train)
		mpe_losses_train_per_split.append(mpe_losses_train)

		losses_per_split.append(losses)
		man_losses_per_split.append(man_losses)
		res_losses_per_split.append(res_losses)
		mpe_losses_per_split.append(mpe_losses)

		print(f'Split {s_id} completed. ', end='')

	# ============================================

	# return (this also moves the data to the CPU)
	return (params_per_split,
			np.array(losses_train_per_split), np.array(man_losses_train_per_split), np.array(res_losses_train_per_split), np.array(mpe_losses_train_per_split),
			np.array(losses_per_split), np.array(man_losses_per_split), np.array(res_losses_per_split), np.array(mpe_losses_per_split))

In [ ]:
def train_models(model, filenames, states, markers, traces, hyperparams=None, save_name=None):
	models_params = list()
	models_losses, models_man_losses, models_res_losses, models_mpe_losses = list(), list(), list(), list()
	models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train = list(), list(), list(), list()

	for i in range(len(filenames)):
		print(f'\n{i+1}/{len(filenames)}: {filenames[i]}... ', end='')

		splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces[i].shape[0]), states[i][['still']].values))

		(params_per_split,
		 losses_train_per_split, man_losses_train_per_split, res_losses_train_per_split, mpe_losses_train_per_split,
		 losses_per_split, man_losses_per_split, res_losses_per_split, mpe_losses_per_split
		 ) = train_model(model, splits,
						 traces[i].values,
						 markers[i].values,
						 states[i].values,
						 hyperparams)

		models_params.append(params_per_split)

		models_losses_train.append(losses_train_per_split)
		models_man_losses_train.append(man_losses_train_per_split)
		models_res_losses_train.append(res_losses_train_per_split)
		models_mpe_losses_train.append(mpe_losses_train_per_split)

		models_losses.append(losses_per_split)
		models_man_losses.append(man_losses_per_split)
		models_res_losses.append(res_losses_per_split)
		models_mpe_losses.append(mpe_losses_per_split)

	if save_name is not None:
		train_utils.save_params(filenames, models_params, save_name)
		train_utils.save_losses(filenames, [
			np.array([models_losses_train[i], models_man_losses_train[i], models_res_losses_train[i], models_mpe_losses_train[i],
					  models_losses[i], models_man_losses[i], models_res_losses[i], models_mpe_losses[i]]) for i in range(len(filenames))
		], save_name)

	return (models_params,
			models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train,
			models_losses, models_man_losses, models_res_losses, models_mpe_losses)

# read & prep

In [ ]:
ll = f'll{int(default_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	# 'vb_reg_lambda': default_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['bom_rs']['means']['vb_reg_lambda']['geometric'],
	# 'wb_reg_lambda': default_hyperparameters['wb_reg_lambda'],
	'wb_reg_lambda': swept_hyperparameters[ll]['bom_rs']['means']['wb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	# 'vb_reg_lambda': default_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['vb_reg_lambda']['geometric'],
	# 'wb_reg_lambda': default_hyperparameters['wb_reg_lambda'],
	'wb_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['wb_reg_lambda']['geometric'],
	# 'wm_reg_lambda': default_hyperparameters['wm_reg_lambda'],
	'wm_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['wm_reg_lambda']['geometric'],
}

In [ ]:
read_ids = list(range(0, 12+1))
# read_ids = [0,	# paper
# 			1,	# nice states
# 			2,	# nice manifold
# 			5,	# variety
# 			6,	# nice quiescence
# 			7,	# nice states
# 			9,	# nice heatmap
# 			11]	# variety
# read_ids = [0]
plot_id = 0

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'brain')

(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# # remove unused discrete state labels
# states = [ds[selection_bhl] for ds in states]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep

def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

markers_grad = [pd.DataFrame(np.gradient(marker.values, axis=0), columns=marker.columns) for marker in markers]

In [ ]:
limit_n = 5
selection_bhl = ['still', 'walk', 'front-groom', 'back-groom', 'abdomen-move']

plot_idx = file_ids.index(plot_id)

# states = [ds[selection_bhl] for ds in states]

# GCAMP

In [ ]:
plot_utils.plot_traces(traces[plot_idx].iloc[:, :limit_n],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] gcamp traces', ylimits=None)

### BOM

In [ ]:
(boms_params,
 boms_losses_train, boms_man_losses_train, boms_res_losses_train, boms_mpe_losses_train,
 boms_losses_test, boms_man_losses_test, boms_res_losses_test, boms_mpe_losses_test,
 ) = train_models(bom,
				  filenames,
				  states,
				  markers,
				  traces,
				  hyperparams=bom_reg_lambdas,
				  save_name=os.path.join('fly-2023-schaffer', ll, 'new',f'ss_bom_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}'))

### HGM

In [ ]:
(hgms_params,
 hgms_losses_train, hgms_man_losses_train, hgms_res_losses_train, hgms_mpe_losses_train,
 hgms_losses_test, hgms_man_losses_test, hgms_res_losses_test, hgms_mpe_losses_test,
 ) = train_models(hgm,
				  filenames,
				  states,
				  markers,
				  traces,
				  hyperparams=hgm_reg_lambdas,
				  save_name=os.path.join('fly-2023-schaffer', ll, 'new',f'ss_hgm_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}'))

### losses

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_train, boms_losses_train,
							  title='gcamp hgm vs bom: train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_man_losses_train, boms_man_losses_train,
							  title='gcamp hgm vs bom: man train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_res_losses_train, boms_res_losses_train,
							  title='gcamp hgm vs bom: res train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_train, boms_mpe_losses_train,
							  title='gcamp hgm vs bom: mpe train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_test, boms_losses_test,
							  title='gcamp hgm vs bom: test',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_man_losses_test, boms_man_losses_test,
							  title='gcamp hgm vs bom: man test',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_res_losses_test, boms_res_losses_test,
							  title='gcamp hgm vs bom: res test',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_test, boms_mpe_losses_test,
							  title='gcamp hgm vs bom: mpe test',
							  ylimits=None)